# Práctica de RAG sobre la Ley 21.526 — caso de uso real (legal)

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase03/notebooks/rag/practica_legal.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Objetivo:** aplicar el pipeline RAG con LangChain (`EnsembleRetriever` con RRF + cross-encoder reranker) sobre un corpus **real, extenso y técnico**: la Ley argentina 21.526 de Entidades Financieras (~91k chars, 67 artículos).
>
> **¿Por qué esta notebook?** En `practica_completa.ipynb` vimos que sobre un corpus chico (programa de cátedra, 5 docs) el reranker **no daba ganancia clara** sobre hybrid+RRF. Acá probamos sobre un corpus realista (legal, terminología técnica) y vemos cuándo el reranker **sí** agrega valor.
>
> **Estructura:**
> 1. Setup — instalar deps, configurar GROQ.
> 2. Descargar y parsear la Ley 21.526 desde una URL pública.
> 3. Chunking por artículo (regex que respeta el formato legal).
> 4. Construir retrievers: dense, BM25, hybrid (RRF), hybrid + reranker.
> 5. **Comparación cualitativa de 3 casos** con análisis del por qué de cada resultado.
> 6. Vista panorámica con benchmark sobre 15 queries.
> 7. Cierre — qué te llevás.
>
> **Tiempo:** ~25 min en Colab (incluyendo descarga del reranker, ~280 MB).

## 0. Setup

In [ ]:
!pip install -q sentence-transformers chromadb rank_bm25 groq numpy pandas matplotlib pypdf requests langchain langchain-community langchain-classic langchain-chroma langchain-huggingface


In [ ]:
# OPCIONAL — sólo para correr en local. En Colab esta celda se autodetecta y se omite
# (en Colab configurá GROQ_API_KEY desde "Secrets" en la barra lateral izquierda).

try:
    import google.colab  # noqa: F401
    print('ℹ️  Colab detectado — saltando .env (usá Colab Secrets para GROQ_API_KEY).')
except ImportError:
    try:
        from dotenv import load_dotenv, find_dotenv
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'])
        from dotenv import load_dotenv, find_dotenv
    env_path = find_dotenv(usecwd=True)
    if env_path:
        load_dotenv(env_path)
        print(f'✓ .env cargado desde {env_path}')
    else:
        print('ℹ️  No se encontró .env. Asegurate de exportar GROQ_API_KEY como env var antes de lanzar jupyter.')

In [ ]:
import os

# En Colab podés usar Secrets ("llave" en la barra lateral)
try:
    from google.colab import userdata
    if 'GROQ_API_KEY' not in os.environ:
        try:
            os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
        except Exception:
            pass
except ImportError:
    pass

assert os.environ.get('GROQ_API_KEY'), (
    'Falta GROQ_API_KEY. Configurala en Colab Secrets o como env var.'
)
print('✓ GROQ_API_KEY configurada')

In [ ]:
from groq import Groq

_groq_client = Groq(api_key=os.environ['GROQ_API_KEY'])


def llamar_llm(messages, model='llama-3.1-8b-instant', temperature=0.1, max_tokens=1024):
    """Wrapper de Groq. Temperature baja para reproducibilidad pedagógica."""
    resp = _groq_client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens,
    )
    return resp.choices[0].message.content


print(llamar_llm([{'role': 'user', 'content': 'Decime "ok" si me podés escuchar'}]))

## 1. Descargar y parsear la Ley 21.526

Bajamos el PDF desde la web pública de la OEA y extraemos el texto con `pypdf`. La ley regula a las entidades financieras en Argentina — bancos comerciales, de inversión, hipotecarios, compañías financieras, cajas de crédito, etc. Tiene 67 artículos agrupados en Títulos y Capítulos.

**¿Por qué este corpus para una clase de RAG?**

- **Tamaño realista** — ~91k chars vs los ~6k del programa de cátedra. El BM25 tiene más vocabulario, el dense tiene más vecinos semánticos, hay margen real para que las técnicas de retrieval se distingan.
- **Terminología técnica** — el lenguaje legal usa palabras como "revocación", "intermediación habitual", "obligación de guardar secreto" que NO son las que un usuario común usaría ("¿cómo cancelan un banco?", "¿pueden contar mis movimientos?"). Esto fuerza al retrieval a hacer trabajo semántico real.
- **Documento público y citable** — no hay problemas de privacidad y cualquiera puede verificar las respuestas contra el texto original.

In [ ]:
import os
import requests

PDF_PATH = 'ley21526.pdf'
# Fuente original (puede 403/timeout): https://www.oas.org/juridico/PDFs/mesicic3_arg_ley21526.pdf
# Mirrors confiables en este mismo repo — probamos main primero, después la branch de desarrollo.
URLS = [
    'https://raw.githubusercontent.com/jorgeroa/ia-utn-frsf/main/clase03/notebooks/rag/ley21526.pdf',
    'https://raw.githubusercontent.com/jorgeroa/ia-utn-frsf/feat/clase03/clase03/notebooks/rag/ley21526.pdf',
]

if not os.path.exists(PDF_PATH):
    last_error = None
    for url in URLS:
        try:
            print(f'Descargando PDF desde {url}...')
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            with open(PDF_PATH, 'wb') as f:
                f.write(r.content)
            print(f'✓ Descargado: {os.path.getsize(PDF_PATH)} bytes')
            break
        except requests.HTTPError as e:
            print(f'  (404 en esta URL, probando la siguiente...)')
            last_error = e
    else:
        raise RuntimeError(f'No se pudo descargar el PDF desde ningún mirror: {last_error}')
else:
    print(f'✓ PDF local encontrado: {os.path.getsize(PDF_PATH)} bytes')

from pypdf import PdfReader

reader = PdfReader(PDF_PATH)
full_text = "\n".join(page.extract_text() for page in reader.pages)
print(f'✓ {len(reader.pages)} páginas, {len(full_text)} caracteres extraídos')
print('\nPrimeras líneas:')
print(full_text[:400])


## 2. Chunking por artículo

Los textos legales tienen una estructura natural: cada artículo es una unidad lógica autocontenida. Chunkeamos por artículo en lugar de por oración o tamaño fijo.

**Trampa visible:** algunos artículos tienen un footnote marker entre el número y el guión:

```
Art. 4 (1) – Autoridad. Funciones*.
Art. 35 (2) – Régimen sancionatorio.
```

Si el regex no contempla el `(1)`, el artículo se "pega" al chunk anterior y se pierde. Esto pasó en una versión previa de esta misma práctica — la query *"¿qué autoridad regula?"* devolvía el Art 3 en lugar del Art 4 porque el contenido del 4 estaba embebido dentro del 3. Es **un error sutil que sólo se ve cuando comparás retrieval contra la verdad del corpus**.

> Moraleja pre-RAG: tu chunking sólo es bueno hasta donde testeás. Mirá el corpus, no asumas que tu regex anda.

In [ ]:
import re

# Regex: split antes de "Art. N -" o "Art. N (footnote) -"
parts = re.split(r'(?=Art\.\s*\d+(?:\s*\([^)]+\))?\s*[\u2013\u2014\-])', full_text)
parts = [p.strip() for p in parts if p.strip()]

from langchain_core.documents import Document

docs = []
for p in parts:
    m = re.match(r'Art\.\s*(\d+)(?:\s*\([^)]+\))?\s*[\u2013\u2014\-]\s*([^\.\n]+?)\.?', p)
    if m:
        art_num = int(m.group(1))
        titulo = m.group(2).strip()[:60]
        body = p[:1800]  # cap article body length
        docs.append(Document(
            page_content=body,
            metadata={'art': art_num, 'titulo': titulo},
        ))

print(f'✓ {len(docs)} artículos chunkeados')
art_nums = sorted({d.metadata['art'] for d in docs})
print(f'  Rango: Art {art_nums[0]} - Art {art_nums[-1]}')
print(f'  Total únicos: {len(art_nums)}')
print()
print('Primeros 5 artículos:')
for d in docs[:5]:
    print(f'  Art {d.metadata["art"]:2}: {d.metadata["titulo"]}')

## 3. Embeddings + ChromaDB

Usamos el **mismo embedding** que en `practica_completa.ipynb`: `paraphrase-multilingual-MiniLM-L12-v2`. Es chico (~85 MB), multilingual, y suficiente — el embedding mejor (ej. e5-large) no aporta mejoras dramáticas con un chunking correcto, y agrega 2 GB de descarga.

> **Decisión consciente**: en producción real para legal-es probablemente querrías un embedding fine-tuned o más grande. Para esta clase priorizamos baja fricción de setup.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

emb = HuggingFaceEmbeddings(model_name='paraphrase-multilingual-MiniLM-L12-v2')
vs = Chroma.from_documents(docs, emb, collection_name='ley21526')
print(f'✓ {vs._collection.count()} chunks indexados en ChromaDB')

## 4. Retrievers: dense, BM25, hybrid (RRF), hybrid + reranker

Construimos los 4 retrievers para poder compararlos:

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# Dense retriever (sólo embeddings)
dense = vs.as_retriever(search_kwargs={'k': 3})

# BM25 (sólo léxico)
bm25 = BM25Retriever.from_documents(docs)
bm25.k = 3

# Hybrid con RRF (BM25 + dense, fusión por ranks)
hybrid = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5])

# Para reranking, subimos k a 10 en cada uno (más candidatos para el cross-encoder)
bm25_rk = BM25Retriever.from_documents(docs); bm25_rk.k = 10
dense_rk = vs.as_retriever(search_kwargs={'k': 10})
hybrid_rk = EnsembleRetriever(retrievers=[bm25_rk, dense_rk], weights=[0.5, 0.5])

ce = HuggingFaceCrossEncoder(model_name='BAAI/bge-reranker-base')
reranker = CrossEncoderReranker(model=ce, top_n=3)
hybrid_rerank = ContextualCompressionRetriever(
    base_compressor=reranker, base_retriever=hybrid_rk
)

print('✓ Retrievers listos: dense, bm25, hybrid, hybrid_rerank')

In [ ]:
SYSTEM_LEGAL = (
    'Sos un asistente legal experto en la Ley argentina 21.526 de Entidades Financieras. '
    'Respondé la pregunta usando ÚNICAMENTE el contexto provisto. '
    'IMPORTANTE: leé TODOS los artículos del contexto e integrá la información relevante de cada uno. '
    'Citá explícitamente el número de artículo cuando corresponda. '
    'Sé concreto, citá el texto, máximo 6 oraciones. '
    'Si el contexto no alcanza, decilo brevemente.'
)


def rag_legal(query, retriever):
    """RAG query sobre Ley 21.526 — devuelve respuesta + artículos retrieved."""
    ds = retriever.invoke(query)
    if len(ds) > 6:
        ds = ds[:6]
    arts = [d.metadata['art'] for d in ds]
    contexto = '\n\n'.join(f'[Art {d.metadata["art"]}] {d.page_content}' for d in ds)
    resp = llamar_llm([
        {'role': 'system', 'content': SYSTEM_LEGAL},
        {'role': 'user', 'content': f'Contexto:\n{contexto}\n\nPregunta: {query}'},
    ])
    return resp, arts


print('✓ rag_legal() listo')

## 5. Comparación cualitativa — 3 casos concretos

En lugar de reportar un porcentaje global, vamos a **inspeccionar 3 queries específicas** comparando hybrid (RRF) vs hybrid + reranker:

| Caso | Patrón | Por qué importa |
|---|---|---|
| **1** | Rerank gana | El cross-encoder rescata un artículo que el hybrid dejó afuera. |
| **2** | Hybrid gana | El reranker filtra de más y descarta el artículo correcto. |
| **3** | Ambos fallan | El vocabulario del usuario no se cruza con el del corpus. |

> **¿Por qué no aggregate score?** La métrica de keywords cuenta presencia literal — está mal calibrada para comparar técnicas. Para evaluación seria de un sistema RAG, ver **Clase 3b** (LLM-as-judge, RAGAS, anotación humana).

In [ ]:
def comparar(titulo, query, expected_arts):
    """Corre hybrid y hybrid+rerank para una query e imprime side-by-side."""
    print('═' * 95)
    print(f'  {titulo}')
    print('═' * 95)
    print(f'Query:    "{query}"')
    print(f'Esperado: artículos {expected_arts}')
    print()

    resp_h, arts_h = rag_legal(query, hybrid)
    resp_r, arts_r = rag_legal(query, hybrid_rerank)

    hits_h = [a for a in arts_h if a in expected_arts]
    hits_r = [a for a in arts_r if a in expected_arts]

    print(f'── lc_hybrid (RRF) ──')
    print(f'   sources: {arts_h}    artículos esperados encontrados: {hits_h}')
    print(f'   resp:')
    for line in resp_h.split('\n'):
        print(f'     {line}')
    print()

    print(f'── lc_hybrid + rerank ──')
    print(f'   sources: {arts_r}    artículos esperados encontrados: {hits_r}')
    print(f'   resp:')
    for line in resp_r.split('\n'):
        print(f'     {line}')


print('✓ comparar() listo')

### Caso 1 — Rerank gana

**Query:** *"¿Quiénes están comprendidos por la ley de entidades financieras?"*

**Respuesta correcta** está en los Arts 1 y 2:
- Art 1 — describe el **sujeto** (personas/entidades que hacen intermediación entre oferta y demanda de recursos financieros).
- Art 2 — enumera las **clases** específicas (bancos comerciales, de inversión, hipotecarios, etc.).

**Predicción:** el hybrid trae el Art 2 (porque BM25 matchea "entidades financieras" → tiene la lista), pero **no trae el Art 1**. El reranker, con más candidatos a evaluar (k=10 de cada retriever) y scoring de relevancia semántica, debería rescatar también el Art 1.

Lo que nos importa observar: si **rerank devuelve una respuesta más completa** porque tiene contexto más completo.

In [ ]:
comparar(
    'CASO 1 — Rerank gana: ¿quiénes están comprendidos por la ley?',
    '¿Quiénes están comprendidos por la ley de entidades financieras?',
    expected_arts=[1, 2],
)

print()
print('💡 Qué mirar:')
print('   El Art 1 dice: "personas o entidades privadas o públicas que realicen')
print('   intermediación habitual entre la oferta y la demanda de recursos financieros".')
print('   El Art 2 lista las clases (bancos, compañías financieras, cajas de crédito).')
print('   Si la respuesta NO menciona la definición conceptual del Art 1, le falta el QUÉ;')
print('   sólo tiene el listado.')

### Caso 2 — Hybrid gana

**Query:** *"¿Qué información se considera secreta por las entidades financieras?"*

**Respuesta correcta** está en los Arts 39 y 40 (obligación de guardar secreto y excepciones).

**Predicción:** el hybrid (con k=3 de cada retriever, ~5 chunks únicos) trae los Arts 39 y 40 directamente — BM25 matchea "secreta" y dense matchea "información" + contexto. El reranker, al filtrar de 20 candidatos a 3, **descarta** uno de esos por priorizar artículos cercanos (Art 37 — funcionarios designados para inspección) que tienen mayor similaridad léxica con la query pero menor relevancia conceptual.

In [ ]:
comparar(
    'CASO 2 — Hybrid gana: ¿qué información se considera secreta?',
    '¿Qué información se considera secreta por las entidades financieras?',
    expected_arts=[39, 40],
)

print()
print('💡 Qué mirar:')
print('   Si la respuesta menciona "operaciones pasivas", "depósitos" o "el secreto NO se')
print('   levanta sin orden judicial" → está usando los Arts 39/40 (correctos).')
print('   Si habla de "funcionarios designados" y "acceso a contabilidad" → está usando')
print('   el Art 37, que es sobre INSPECCIÓN, no sobre SECRETO. Es relevante pero no responde.')

### Caso 3 — Ambos fallan (con un giro peligroso)

**Query:** *"¿Cuál es el procedimiento para revocar la autorización para funcionar?"*

**Respuesta correcta** está en los Arts 44 y 45 (revocación y trámite específicos).

**Predicción:** ni hybrid ni rerank traen los Arts 44/45 al top-3. ¿Por qué?

- **BM25** busca "revocar autorización" exacto, pero la ley usa *"causales de exclusión"*, *"suspensión transitoria"*, *"cesación de actividad"* en esos artículos. Match léxico cero.
- **Dense (embedding chico, no fine-tuneado para legal-es)** asocia "revocar autorización" con "autorización para operar" — y trae Arts sobre AUTORIZACIÓN INICIAL (Arts 7, 8, 15), no sobre revocación específica.
- **Reranker** trabaja sobre lo que el bi-encoder le da. Si el Art 44/45 ni siquiera está en los top-20 candidatos, el reranker no lo puede rescatar.

**El giro peligroso (mirá las respuestas):** aunque la retrieval falla en traer los Arts 44/45, **el LLM va a producir respuestas plausibles** citando artículos relacionados (Art 15 que tangencialmente menciona revocación, Art 47 sobre apelación). Un usuario sin conocimiento legal experto **no podrá distinguir** que falta la regulación principal del procedimiento.

> **Esto es exactamente por qué importa la evaluación end-to-end (Clase 3b)** — no alcanza con "el LLM contestó algo coherente". Necesitás validar contra ground truth o anotación humana. Un sistema RAG que cita el artículo equivocado es **peor** que uno que admite no saber.


In [ ]:
comparar(
    'CASO 3 — Ambos fallan: ¿cuál es el procedimiento para revocar la autorización?',
    '¿Cuál es el procedimiento para revocar la autorización para funcionar?',
    expected_arts=[44, 45],
)

print()
print('💡 Qué mirar:')
print('   Mirá los "sources" — los artículos retrieved. Si ninguno menciona específicamente')
print('   "revocación de la autorización", el retrieval falló. Es probable que la respuesta')
print('   del LLM sea genérica ("el BC tiene facultades para...") o admita no saber.')
print('   Tareas para casa: probá embeddings fine-tuneados para legal-es, query expansion,')
print('   o agregá los títulos de los artículos como metadata adicional al chunk.')

## 6. Vista panorámica — 15 queries

Para confirmar que el patrón "rerank y hybrid son complementarios, no jerárquicos" no es anecdótico, corremos 15 queries y vemos quién acierta a traer **algún artículo esperado** al top-3.

No corremos el LLM acá — sólo retrieval, mucho más rápido.

In [ ]:
QUERIES = [
    ('q1',  '¿Qué autoridad regula las entidades financieras y qué facultades tiene?', [4]),
    ('q2',  '¿Qué requisitos debe cumplir una persona para ser director o síndico?', [10]),
    ('q3',  '¿Cuáles son las operaciones permitidas a un banco de inversión?', [22]),
    ('q4',  '¿Qué pasa si una entidad financiera viola el secreto bancario?', [39, 40, 41]),
    ('q5',  '¿Cuál es el procedimiento para revocar la autorización para funcionar?', [44, 45]),
    ('q6',  '¿Qué obligaciones de información tienen las entidades hacia el Banco Central?', [36, 37]),
    ('q7',  '¿Cuándo procede la liquidación de una entidad financiera?', [49, 50]),
    ('q8',  '¿Qué garantías existen sobre los depósitos bancarios?', [56]),
    ('q9',  '¿Quiénes están comprendidos por la ley de entidades financieras?', [1, 2]),
    ('q10', '¿Cómo se aplican las sanciones a una entidad infractora?', [41, 42]),
    ('q11', '¿Qué operaciones puede realizar un banco comercial?', [21]),
    ('q12', '¿Qué normas se aplican a las cajas de crédito?', [26]),
    ('q13', '¿Qué pasa con los depositantes en caso de quiebra de una entidad?', [53, 54, 55]),
    ('q14', '¿Cómo se realiza el control de las operaciones por el Banco Central?', [37, 38]),
    ('q15', '¿Qué información se considera secreta por las entidades financieras?', [39, 40]),
]

import pandas as pd

rows = []
for qid, query, expected in QUERIES:
    h_arts = [d.metadata['art'] for d in hybrid.invoke(query)[:3]]
    r_arts = [d.metadata['art'] for d in hybrid_rerank.invoke(query)]
    h_hits = len([a for a in h_arts if a in expected])
    r_hits = len([a for a in r_arts if a in expected])

    # Verdict: contamos el numero de articulos esperados en top-3, no solo "al menos uno".
    # Asi una query con expected=[1,2] donde hybrid trae solo el 2 y rerank trae 1 y 2
    # se cuenta como "rerank gana" (mas coverage), no como empate.
    if r_hits > h_hits: verdict = '⭐ rerank gana'
    elif h_hits > r_hits: verdict = '⚠️ hybrid gana'
    elif h_hits > 0: verdict = '✓ empate (ambos)'
    else: verdict = '✗ ambos fallan'

    rows.append({
        'id': qid,
        'query': query[:60] + ('...' if len(query) > 60 else ''),
        'esperado': str(expected),
        'h_hits': f'{h_hits}/{len(expected)}',
        'r_hits': f'{r_hits}/{len(expected)}',
        'hybrid (top-3)': str(h_arts),
        'rerank (top-3)': str(r_arts),
        'resultado': verdict,
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print()

# Resumen
counts = df['resultado'].value_counts()
print('Resumen:')
for k, v in counts.items():
    print(f'  {k}: {v}')


## 7. Cierre — qué te llevás

### Datos crudos del benchmark

Sobre 15 queries en un corpus legal real:

- **El reranker gana cleanly en algunas queries** (típicamente 2-3 de 15) — específicamente cuando el artículo correcto está en los rangos #4–#10 del retrieval bi-encoder y el cross-encoder lo identifica.
- **El hybrid (sólo RRF) gana cleanly en otras** (típicamente 4-5 de 15) — cuando filtrar a 3 candidatos descarta artículos que el LLM hubiera usado bien.
- **En el resto**, o ambos aciertan, o ambos fallan.

### Lecciones para producción

1. **No defaultes al reranker porque sí.** Tiene costo (latencia + ~280 MB de modelo + más complejidad). En corpus chicos puede empeorar el resultado.
2. **El reranker brilla cuando**: corpus mediano-grande (cientos+ de chunks), queries semánticamente complejas (multi-hop), Y el LLM necesita el chunk *exactamente correcto* en el top-3.
3. **Si el embedding base es malo para tu dominio, ningún reranker te salva** — los Arts 44-45 nunca llegan al top-20, por lo que el reranker no puede rescatarlos.
4. **El chunking importa MÁS que la elección del reranker** — recordá el bug del Art 4 con `(1)`.

### Decisión honesta sobre rerank en producción

```
                    pequeño        mediano        grande
                    (<50 chunks)   (100-1k)       (10k+)
  ──────────────────────────────────────────────────────
  rerank vale       no necesario   evaluá          casi siempre
                    (ya alcanza    en TU corpus    (RRF rinde
                    con hybrid)    con métricas    saturado)
                                   reales
```

### Siguiente paso

- **Clase 3b** te muestra **cómo medir** seriamente: LLM-as-judge, RAGAS (faithfulness, answer relevance, context precision/recall), Arize AX para observability end-to-end.
- Una vez que tenés métricas, las preguntas "¿uso rerank?" "¿cambio embedding?" "¿chunking semántico?" se vuelven decisiones de datos, no de fe.